# Amharic byte-level Mamba vs. Transformer

Hypothesis: a byte-level Mamba (linear-time SSM) achieves better compute
efficiency (wall-clock, memory) than a parameter-matched byte-level
Transformer on Amharic text, at matched step budget — because standard
tokenizers measurably break down for Amharic (the "Token Tax" finding:
Ge'ez script gets tokenized into many more pieces per word than
comparable-resource Latin-script languages), so removing tokenization
should help specifically here.

Three-way comparison to properly isolate both variables (architecture and
tokenization), not just one pairwise comparison:
1. **byte-Mamba** — the proposed method
2. **byte-Transformer** — isolates the architecture effect (same input representation as #1)
3. **tokenized-Transformer** (SentencePiece/BPE) — isolates the tokenization effect (same architecture as #2)

Scope, stated honestly: a proof-of-concept at small model/data scale, not a
competitive model — see the write-up checklist at the end for what this
does and doesn't establish.

In [ ]:
import sys
# Try to make stdout line-buffered so prints show in the Kaggle log in real
# time. NOTE: this only works in plain CPython. On Kaggle the notebook runs
# under IPython (via papermill), where sys.stdout is an IPython `OutStream`
# that has NO .reconfigure() method - calling it there raises
# AttributeError and crashed the whole notebook on line 1 in a previous run.
# So it's guarded: if reconfigure isn't available, we simply rely on
# flush=True at the actual long-running progress prints instead (added in
# the tokenizer and training cells), which is the mechanism that actually
# works inside IPython.
try:
    sys.stdout.reconfigure(line_buffering=True)
except (AttributeError, ValueError):
    pass

!pip install -q datasets --upgrade
# torch is intentionally NOT upgraded here. Kaggle's GPU image ships a
# torch build already matched to its driver and to the pre-installed
# torchvision/torchaudio/CUDA toolkit. `--upgrade` on torch pulled in
# torch 2.13.0 plus a fresh CUDA 13 runtime (large nvidia-* wheel
# downloads), which conflicted with the image's actual CUDA 12.x driver
# stack - pip's own resolver flagged this directly (torchvision/torchaudio
# needing torch==2.10.0, cuda-toolkit 13.0.3.0 vs 12.* needed elsewhere).
# In a real run this caused a silent hang: the log stopped completely
# right after data collection, before even the plain print() at the start
# of the next cell, for 40+ minutes - consistent with `import torch`
# stalling on a mismatched CUDA runtime, a known failure mode. Only
# `datasets` actually needs the newer version here.

## 1. Data — byte-level Amharic corpus

Pulls every source that loads successfully (each wrapped in try/except, so
one bad dataset config doesn't take down the whole pipeline). Each document
is cleaned and filtered individually before being added to the corpus:

- **Control characters stripped** (null bytes, other non-printable
  characters that can appear in scraped web text and would otherwise
  corrupt byte-level training).
- **Too-short documents dropped** (empty/near-empty rows are noise, not
  signal).
- **Non-Amharic documents dropped** via a Ge'ez-script character ratio
  check — the "am"-labeled streams (especially C4 and GlotCC, both sourced
  from noisy Common Crawl language-ID) can and do leak in wrong-language
  content; this catches that rather than silently training on it.
- **Exact-duplicate documents dropped** via hashing — repeated boilerplate
  is a well-documented problem in web-scraped corpora (this is explicitly
  why the original C4 paper does deduplication, and why MiniPile's quality
  curation outperformed raw volume in our own literature review).

Deliberately NOT doing embedding-based semantic deduplication or
cluster-based quality filtering (MiniPile's actual method) — that requires
its own embedding model and clustering step, which is disproportionate
machinery for this corpus size and would introduce its own new failure
surface. This is a stated scope boundary, not a silent omission: exact-hash
dedup + language-ratio + control-char filtering catches the most damaging,
cheapest-to-catch problems; semantic-level curation is real future work.

No tokenizer at this stage either way: raw UTF-8 bytes, vocab = 256. If a
source fails, the printed error names the exact repo/config to check on the
HF Hub.

In [ ]:
import os, time, re, hashlib, gc
import numpy as np
from datasets import load_dataset

OUT_DIR = "/kaggle/working/data" if os.path.exists("/kaggle/working") else "./data"
os.makedirs(OUT_DIR, exist_ok=True)

# QUICK_TEST_MODE = False skips the two slow/large sources (allenai/c4, GlotCC)
# and keeps only the fast small ones (~64MB, under a minute) for validating the
# whole pipeline cheaply. Set to False for the real full-corpus run. NOTE on
# size: the full run pulls C4 (~1.2GB) + GlotCC + the small sources - realistic
# total is ~1.5-2GB, NOT the 4GB cap below. A true 4GB Amharic corpus needs the
# UnifiedCrawl multi-archive Common-Crawl pipeline (a separate ~1-day effort),
# not a HuggingFace download. The 4GB cap just means "take everything these
# sources have"; it will stop naturally when the sources run out, well short of 4GB.
QUICK_TEST_MODE = False

MAX_TOTAL_BYTES = 4_000_000_000

GEEZ_LO, GEEZ_HI = 0x1200, 0x137F  # Ethiopic Unicode block (Amharic, Tigrinya, Ge'ez)
CONTROL_CHAR_RE = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]')
MIN_CHARS = 50
MIN_GEEZ_RATIO = 0.3

seen_hashes = set()
clean_stats = {"kept": 0, "too_short": 0, "not_amharic": 0, "duplicate": 0}

def geez_ratio(text):
    if not text:
        return 0.0
    n_geez = sum(1 for c in text if GEEZ_LO <= ord(c) <= GEEZ_HI)
    return n_geez / len(text)

def clean_and_filter(text):
    """Returns cleaned text, or None if the document should be dropped."""
    if not text:
        clean_stats["too_short"] += 1
        return None
    text = CONTROL_CHAR_RE.sub('', text).strip()
    if len(text) < MIN_CHARS:
        clean_stats["too_short"] += 1
        return None
    if geez_ratio(text) < MIN_GEEZ_RATIO:
        clean_stats["not_amharic"] += 1
        return None
    h = hashlib.md5(text.encode("utf-8")).hexdigest()
    if h in seen_hashes:
        clean_stats["duplicate"] += 1
        return None
    seen_hashes.add(h)
    clean_stats["kept"] += 1
    return text

SMALL_SOURCES = [
    ("wikimedia/wikipedia", "20231101.am", "train", "text"),
    ("masakhane/masakhanews", "amh", "train", "text"),
    ("masakhane/masakhanews", "amh", "validation", "text"),
    ("masakhane/masakhanews", "amh", "test", "text"),
]

t0 = time.time()
total_bytes = 0
corpus_path = os.path.join(OUT_DIR, "corpus.txt")

if QUICK_TEST_MODE:
    print("QUICK_TEST_MODE is ON - only pulling Wikipedia, MasakhaNews, and xlsum "
          "(skipping allenai/c4 and GlotCC). Set QUICK_TEST_MODE = False above for the full corpus.\n", flush=True)
else:
    print("FULL-CORPUS MODE - pulling all sources incl. allenai/c4 and GlotCC. "
          "Expect ~1.5-2GB total and several minutes of streaming.\n", flush=True)

# Stream directly to corpus.txt to prevent memory bloat/OOM
with open(corpus_path, "w", encoding="utf-8") as f_corpus:
    for repo_id, config, split, field in SMALL_SOURCES:
        try:
            ds = load_dataset(repo_id, config, split=split)
            n_added = 0
            for row in ds:
                t = clean_and_filter(row.get(field, ""))
                if t:
                    f_corpus.write(t + "\n")
                    b_len = len(t.encode("utf-8")) + 1
                    total_bytes += b_len
                    n_added += b_len
            print(f"[OK]   {repo_id} ({config}/{split}): {n_added:,} bytes | running total: {total_bytes/1e6:.1f} MB", flush=True)
        except Exception as e:
            print(f"[FAIL] {repo_id} ({config}/{split}): {e}")

    # csebuetnlp/xlsum
    if total_bytes < MAX_TOTAL_BYTES:
        try:
            print("\nPulling xlsum Amharic (parquet branch, data_files glob)...")
            xlsum_ds = load_dataset(
                "csebuetnlp/xlsum",
                data_files={"train": "amharic/train/*.parquet"},
                revision="refs/convert/parquet",
                split="train",
            )
            n_added = 0
            for row in xlsum_ds:
                t = clean_and_filter(row.get("text", ""))
                if t:
                    f_corpus.write(t + "\n")
                    b_len = len(t.encode("utf-8")) + 1
                    total_bytes += b_len
                    n_added += b_len
            print(f"[OK]   csebuetnlp/xlsum (amharic/train): {n_added:,} bytes | running total: {total_bytes/1e6:.1f} MB", flush=True)
        except Exception as e:
            print(f"[FAIL] csebuetnlp/xlsum: {e}")

    # allenai/c4 Amharic (streaming) - skipped entirely in QUICK_TEST_MODE
    if not QUICK_TEST_MODE and total_bytes < MAX_TOTAL_BYTES:
        try:
            print("\nPulling allenai/c4 Amharic (streaming)...")
            c4_am = load_dataset("allenai/c4", "am", split="train", streaming=True)
            c4_bytes = 0
            _c4_last = time.time()
            for row in c4_am:
                t = clean_and_filter(row.get("text", ""))
                if t:
                    f_corpus.write(t + "\n")
                    b_len = len(t.encode("utf-8")) + 1
                    total_bytes += b_len
                    c4_bytes += b_len
                    if time.time() - _c4_last > 15:
                        print(f"  [c4] {c4_bytes/1e6:.0f} MB from C4 so far | total {total_bytes/1e6:.0f}/{MAX_TOTAL_BYTES/1e6:.0f} MB | {time.time()-t0:.0f}s elapsed", flush=True)
                        _c4_last = time.time()
                    if total_bytes >= MAX_TOTAL_BYTES:
                        break
            print(f"[OK]   allenai/c4 (am): {c4_bytes:,} bytes | running total: {total_bytes/1e6:.1f} MB", flush=True)
        except Exception as e:
            print(f"[FAIL] allenai/c4 (am): {e}")

    # GlotCC Amharic - skipped entirely in QUICK_TEST_MODE
    if not QUICK_TEST_MODE and total_bytes < MAX_TOTAL_BYTES:
        glotcc_loaded = False
        print("\nAttempting GlotCC Amharic slice...")
        try:
            glot_ds = load_dataset("cis-lmu/GlotCC-v1", "amh-Ethi", split="train", streaming=True)
            glotcc_loaded = True
            print("  -> loaded via direct per-language config 'amh-Ethi'")
        except Exception as e1:
            print(f"  [attempt 1 FAILED - config 'amh-Ethi']: {e1}")
            try:
                glot_ds = load_dataset("cis-lmu/GlotCC-v1", split="train", streaming=True)
                glot_ds = glot_ds.filter(lambda x: str(x.get("lang", x.get("language", ""))).startswith("amh"))
                glotcc_loaded = True
                print("  -> loaded via streaming + manual language filter (slow path)")
            except Exception as e2:
                print(f"  [attempt 2 FAILED - streaming+filter]: {e2}")

        if glotcc_loaded:
            try:
                glot_bytes = 0
                _glot_last = time.time()
                for row in glot_ds:
                    t = clean_and_filter(row.get("text", row.get("content", "")))
                    if t:
                        f_corpus.write(t + "\n")
                        b_len = len(t.encode("utf-8")) + 1
                        total_bytes += b_len
                        glot_bytes += b_len
                        if time.time() - _glot_last > 15:
                            print(f"  [glotcc] {glot_bytes/1e6:.0f} MB from GlotCC so far | total {total_bytes/1e6:.0f} MB | {time.time()-t0:.0f}s elapsed", flush=True)
                            _glot_last = time.time()
                        if total_bytes >= MAX_TOTAL_BYTES:
                            break
                print(f"[OK]   GlotCC (amh): {glot_bytes:,} bytes | running total: {total_bytes/1e6:.1f} MB", flush=True)
            except Exception as e:
                print(f"[FAIL] GlotCC iteration: {e}")

if total_bytes == 0:
    raise RuntimeError("No sources loaded successfully. Check internet access and HF dataset configurations.")

print(f"\nCleaning stats across all sources: {clean_stats}", flush=True)
n_seen = sum(clean_stats.values())
if n_seen > 0:
    print(f"  kept {clean_stats['kept']:,}/{n_seen:,} documents ({clean_stats['kept']/n_seen*100:.1f}%)", flush=True)

# Stream corpus.txt into train.bin (95%) and val.bin (5%) without keeping the full buffer in RAM
total_file_bytes = os.path.getsize(corpus_path)
split_idx = int(total_file_bytes * 0.95)

train_path = os.path.join(OUT_DIR, "train.bin")
val_path = os.path.join(OUT_DIR, "val.bin")

with open(corpus_path, "rb") as f_in, open(train_path, "wb") as f_tr, open(val_path, "wb") as f_val:
    written = 0
    while True:
        chunk = f_in.read(16 * 1024 * 1024)  # 16MB streaming chunk
        if not chunk:
            break
        chunk_len = len(chunk)
        if written + chunk_len <= split_idx:
            f_tr.write(chunk)
        elif written >= split_idx:
            f_val.write(chunk)
        else:
            to_train = split_idx - written
            f_tr.write(chunk[:to_train])
            f_val.write(chunk[to_train:])
        written += chunk_len

print(f"Total corpus: {total_file_bytes:,} bytes ({total_file_bytes / 1e6:.1f} MB)", flush=True)
print(f"train.bin: {os.path.getsize(train_path):,} bytes | val.bin: {os.path.getsize(val_path):,} bytes", flush=True)
print(f"Done preparing byte data in {time.time() - t0:.1f}s", flush=True)
gc.collect()

## 2. Models — three configurations, matched in parameter count

- **TinyMamba** (byte-level): minimal pure-PyTorch selective SSM. Sequential
  scan, not the fused CUDA kernel from the official `mamba-ssm` package -
  chosen for build simplicity (no CUDA toolchain/nvcc version matching to
  debug), not as a shortcut; the math is verified against the reference
  `mamba-minimal` implementation.
- **TinyTransformer** (byte-level): standard pre-norm causal decoder, same
  depth/width as TinyMamba - isolates the *architecture* variable, since
  both operate on identical byte-level input.
- **TinyTransformer-Tokenized** (SentencePiece/BPE): the same Transformer
  architecture as above, but on standard subword-tokenized input - isolates
  the *tokenization* variable, since both use the same architecture.

Together these three let us attribute any observed difference to
architecture, tokenization, or both - rather than one pairwise comparison
that conflates the two variables the hypothesis actually depends on.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

VOCAB_SIZE = 256  # raw bytes, no special tokens

# ---------------------------------------------------------------------------
# Fused Mamba kernel (mamba-ssm). This is the REAL linear-time Mamba: it
# replaces the slow pure-Python sequential scan with a fused CUDA kernel,
# giving a ~100-200x speedup. It only installs/runs on a proper GPU box with
# a CUDA toolchain (e.g. a rented Ampere/Ada GPU: `pip install causal-conv1d
# mamba-ssm`). If it's not importable (e.g. plain Kaggle), we transparently
# fall back to the pure-PyTorch JIT scan below so the notebook still runs.
# IMPORTANT for the research claim: the wall-clock EFFICIENCY comparison is
# only valid with this fused kernel - the naive scan measures our
# implementation, not the architecture.
# ---------------------------------------------------------------------------
try:
    from mamba_ssm import Mamba as _FusedMamba
    _HAVE_FUSED_MAMBA = True
    print("[mamba] FUSED mamba-ssm kernel available -> fast, valid wall-clock comparison", flush=True)
except Exception as _e:
    _FusedMamba = None
    _HAVE_FUSED_MAMBA = False
    print(f"[mamba] mamba-ssm NOT available ({type(_e).__name__}) -> pure-PyTorch JIT scan fallback (slow; "
          f"wall-clock efficiency numbers would NOT be valid). Install with: pip install causal-conv1d mamba-ssm", flush=True)

# ---------------------------------------------------------------------------
# TinyMamba (fused kernel if available, else JIT-accelerated pure-PyTorch scan)
# ---------------------------------------------------------------------------

@torch.jit.script
def _selective_scan_jit(x_conv: torch.Tensor, delta: torch.Tensor, A: torch.Tensor,
                        Bp: torch.Tensor, Cp: torch.Tensor, D: torch.Tensor) -> torch.Tensor:
    # JIT-compiled fallback scan: fuses the Python loop into TorchScript, but
    # is still fundamentally sequential (used only when mamba-ssm is absent).
    B, L, d_inner = x_conv.shape
    d_state = A.shape[1]

    deltaA = torch.exp(delta.unsqueeze(-1) * A)  # (B, L, d_inner, d_state)
    deltaB_x = delta.unsqueeze(-1) * Bp.unsqueeze(2) * x_conv.unsqueeze(-1)  # (B, L, d_inner, d_state)

    h = torch.zeros(B, d_inner, d_state, device=x_conv.device, dtype=x_conv.dtype)
    ys = []
    for t in range(L):
        h = deltaA[:, t] * h + deltaB_x[:, t]
        y_t = (h * Cp[:, t].unsqueeze(1)).sum(dim=-1)
        ys.append(y_t)
    y = torch.stack(ys, dim=1)
    y = y + x_conv * D
    return y


class MambaBlock(nn.Module):
    """Pure-PyTorch selective SSM (fallback when mamba-ssm isn't installed)."""
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = expand * d_model
        self.d_state = d_state
        self.dt_rank = max(d_model // 16, 1)

        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            groups=self.d_inner, padding=d_conv - 1, bias=True,
        )

        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

    def forward(self, x):
        B, L, _ = x.shape
        xz = self.in_proj(x)
        x_in, res = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_in.transpose(1, 2))[:, :, :L]
        x_conv = F.silu(x_conv.transpose(1, 2))

        x_dbl = self.x_proj(x_conv)
        delta, Bp, Cp = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], dim=-1)
        delta = F.softplus(self.dt_proj(delta))

        A = -torch.exp(self.A_log)
        y = _selective_scan_jit(x_conv, delta, A, Bp, Cp, self.D)
        y = y * F.silu(res)
        return self.out_proj(y)


def make_mamba_mixer(d_model, d_state, d_conv, expand):
    """Fused mamba-ssm kernel if available (fast + valid), else pure-PyTorch fallback."""
    if _HAVE_FUSED_MAMBA:
        return _FusedMamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
    return MambaBlock(d_model, d_state, d_conv, expand)


class TinyMamba(nn.Module):
    def __init__(self, d_model=256, n_layer=6, d_state=16, d_conv=4, expand=2,
                 vocab_size=VOCAB_SIZE):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "norm": nn.LayerNorm(d_model),
                "mixer": make_mamba_mixer(d_model, d_state, d_conv, expand),
            }) for _ in range(n_layer)
        ])
        self.norm_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

    def forward(self, idx, targets=None):
        x = self.embed(idx)
        for layer in self.layers:
            x = x + layer["mixer"](layer["norm"](x))
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


# ---------------------------------------------------------------------------
# TinyTransformer (matched baseline)
# ---------------------------------------------------------------------------

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_head: int):
        super().__init__()
        assert d_model % n_head == 0
        self.n_head = n_head
        self.d_head = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, L, D = x.shape
        qkv = self.qkv(x).view(B, L, 3, self.n_head, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, L, D)
        return self.proj(y)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, d_ff: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_head)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff, bias=False),
            nn.GELU(),
            nn.Linear(d_ff, d_model, bias=False),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class TinyTransformer(nn.Module):
    def __init__(self, d_model=256, n_layer=6, n_head=8, d_ff=None,
                 max_len=1024, vocab_size=VOCAB_SIZE):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_head, d_ff) for _ in range(n_layer)
        ])
        self.norm_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

    def forward(self, idx, targets=None):
        B, L = idx.shape
        pos = torch.arange(L, device=idx.device)
        x = self.embed(idx) + self.pos_embed(pos)[None, :, :]
        for layer in self.layers:
            x = layer(x)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

## 2b. Train a SentencePiece tokenizer, for the third (tokenized) model

This is the piece that actually isolates the tokenization variable. Trained
on the same corpus the byte-level models see, so the *only* difference for
the third model is subword vs. byte input, not different training data.

In [ ]:
!pip install -q sentencepiece

import os, time, gc
import sentencepiece as spm
import numpy as np

OUT_DIR = "/kaggle/working/data" if os.path.exists("/kaggle/working") else "./data"
TOK_VOCAB_SIZE = 16000
SP_PREFIX = os.path.join(OUT_DIR, "amharic_sp")
CORPUS_TXT = os.path.join(OUT_DIR, "corpus.txt")
SP_TRAIN_INPUT = os.path.join(OUT_DIR, "tokenizer_train_sample.txt")
TOK_TRAIN_PATH = os.path.join(OUT_DIR, "tok_train.bin")
TOK_VAL_PATH = os.path.join(OUT_DIR, "tok_val.bin")

# 1. Create bounded sample file for tokenizer training (20MB is ample for a 16k vocab)
TOK_TRAIN_CAP_BYTES = 20_000_000
written = 0
with open(CORPUS_TXT, "r", encoding="utf-8") as f_in, open(SP_TRAIN_INPUT, "w", encoding="utf-8") as f_out:
    for line in f_in:
        f_out.write(line)
        written += len(line.encode("utf-8"))
        if written >= TOK_TRAIN_CAP_BYTES:
            break

print(f"Tokenizer training sample created: {written/1e6:.1f} MB (from {os.path.getsize(CORPUS_TXT)/1e6:.1f} MB corpus)", flush=True)
import time as _t
_tk0 = _t.time()
print(f"[{_tk0:.0f}] ENTERING SentencePieceTrainer.train() now (model_type=unigram) - if the READY line below does not appear within ~1-2 min, this call is the bottleneck.", flush=True)

# Train SentencePiece. Unigram (SentencePiece default) chosen after BPE was
# found to be pathologically slow on this data (stuck >16 min on 20MB even
# with input_sentence_size cap + threads). Unigram trains in seconds and is
# the standard subword algorithm for multilingual models (T5/mBART/XLM-R),
# so it isolates the same subword-vs-byte variable the ablation tests.
spm.SentencePieceTrainer.train(
    input=SP_TRAIN_INPUT,
    model_prefix=SP_PREFIX,
    vocab_size=TOK_VOCAB_SIZE,
    character_coverage=0.9995,
    model_type="unigram",
    hard_vocab_limit=False,
    input_sentence_size=1_000_000,   # cap sentences fed to the trainer (has no
                                     # effect here since the sample has fewer
                                     # lines than this, but harmless and correct
                                     # for the full-corpus run)
    shuffle_input_sentence=True,
    num_threads=os.cpu_count() or 4, # use all cores instead of one
)

sp = spm.SentencePieceProcessor(model_file=f"{SP_PREFIX}.model")
actual_vocab_size = sp.get_piece_size()
print(f"SentencePiece tokenizer READY in {_t.time()-_tk0:.1f}s. Actual Vocab Size: {actual_vocab_size}", flush=True)

# 2. Tokenize corpus line-by-line in batches (fast C++ internal execution, zero hang, minimal RAM)
total_tokens = 0
total_text_bytes = os.path.getsize(CORPUS_TXT)
split_point_bytes = int(total_text_bytes * 0.95)
bytes_processed = 0

t_start = time.time()
last_print = time.time()

with open(CORPUS_TXT, "r", encoding="utf-8") as f_in,      open(TOK_TRAIN_PATH, "wb") as f_tr,      open(TOK_VAL_PATH, "wb") as f_val:
    
    batch_lines = []
    for line in f_in:
        batch_lines.append(line)
        if len(batch_lines) >= 4000:
            encoded_batch = sp.encode(batch_lines, out_type=int)
            for line_str, ids in zip(batch_lines, encoded_batch):
                l_bytes = len(line_str.encode("utf-8"))
                if ids:
                    arr = np.array(ids, dtype=np.int32)
                    if bytes_processed < split_point_bytes:
                        arr.tofile(f_tr)
                    else:
                        arr.tofile(f_val)
                    total_tokens += len(arr)
                bytes_processed += l_bytes
            batch_lines = []
            
            if time.time() - last_print > 15:
                pct = bytes_processed / total_text_bytes * 100
                elapsed = time.time() - t_start
                rate = bytes_processed / elapsed if elapsed > 0 else 0
                eta_s = (total_text_bytes - bytes_processed) / rate if rate > 0 else 0
                print(f"  [tokenizing] {bytes_processed/1e6:.1f}/{total_text_bytes/1e6:.1f} MB ({pct:.1f}%) | "
                      f"{total_tokens:,} tokens | {elapsed:.0f}s elapsed | ETA {eta_s:.0f}s", flush=True)
                last_print = time.time()

    # Flush remaining batch
    if batch_lines:
        encoded_batch = sp.encode(batch_lines, out_type=int)
        for line_str, ids in zip(batch_lines, encoded_batch):
            l_bytes = len(line_str.encode("utf-8"))
            if ids:
                arr = np.array(ids, dtype=np.int32)
                if bytes_processed < split_point_bytes:
                    arr.tofile(f_tr)
                else:
                    arr.tofile(f_val)
                total_tokens += len(arr)
            bytes_processed += l_bytes

BYTES_PER_TOKEN = total_text_bytes / max(1, total_tokens)
train_tokens = os.path.getsize(TOK_TRAIN_PATH) // 4
val_tokens = os.path.getsize(TOK_VAL_PATH) // 4

print(f"\nTokenization finished in {time.time() - t_start:.1f}s!", flush=True)
print(f"tok_train.bin: {train_tokens:,} tokens | tok_val.bin: {val_tokens:,} tokens", flush=True)
print(f"Measured bytes-per-token: {BYTES_PER_TOKEN:.2f}", flush=True)
gc.collect()

In [ ]:
# Sanity check before committing GPU resources: confirm all three configs
# have close parameter counts.
tok_vocab = sp.get_piece_size() if 'sp' in globals() else TOK_VOCAB_SIZE
_mamba_check = TinyMamba(d_model=256, n_layer=6, d_state=16, d_conv=4, expand=2)
_xf_byte_check = TinyTransformer(d_model=256, n_layer=6, n_head=8, vocab_size=VOCAB_SIZE)
_xf_tok_check = TinyTransformer(d_model=256, n_layer=6, n_head=8, vocab_size=tok_vocab)

n_mamba = count_params(_mamba_check)
n_xf_byte = count_params(_xf_byte_check)
n_xf_tok = count_params(_xf_tok_check)
print(f"TinyMamba (byte) params:              {n_mamba:,}")
print(f"TinyTransformer (byte) params:        {n_xf_byte:,}")
print(f"TinyTransformer (tokenized) params:   {n_xf_tok:,}")
print(f"Ratio (byte-xf/mamba):     {n_xf_byte / n_mamba:.3f}")
print(f"Ratio (tok-xf/mamba):      {n_xf_tok / n_mamba:.3f}")
del _mamba_check, _xf_byte_check, _xf_tok_check


## 3. Shared training loop

Generalized to work for all three models: byte-level data (`vocab=256`) or
tokenized data (`vocab=TOK_VOCAB_SIZE`), passed in explicitly per run.

Important correctness point: a tokenized model's loss is nats-per-*token*,
not nats-per-*byte* - directly comparing it to the byte-level models' loss
would be comparing different units. `bits_per_byte()` below takes a
`bytes_per_unit` conversion factor (1.0 for byte-level models, the measured
average bytes-per-token ratio for the tokenized model) so all three land on
the same metric, bits per byte of original text - the fair common currency
regardless of what each model's vocabulary actually is.

`BLOCK_SIZE=512` (in the relevant unit - bytes or tokens) keeps memory in
check for the 6GB-VRAM machine this continues on afterward. Logs bits-per-
byte (train+val), wall-clock, and peak GPU memory every `EVAL_EVERY` steps -
those are the actual result this experiment produces. Checkpoints every
eval, since a cloud notebook session can disconnect and losing a completed
training run to that would be a real problem, not just an inconvenience.

In [ ]:
SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def _pick_device():
    if not torch.cuda.is_available():
        return "cpu"
    try:
        major, minor = torch.cuda.get_device_capability(0)
        dev_sm = major * 10 + minor
        name = torch.cuda.get_device_name(0)
        arch_list = torch.cuda.get_arch_list()
        supported = sorted(int(a[3:]) for a in arch_list if a.startswith("sm_") and a[3:].isdigit())
        ok = (dev_sm in supported) or (supported and dev_sm >= min(supported))
        if not ok:
            print(f"[device] {name} is sm_{dev_sm}, falling back to CPU for compatibility.", flush=True)
            return "cpu"
        _p = torch.randn(64, 64, device="cuda"); _ = (_p @ _p).sum().item()
        print(f"[device] using GPU: {name} (sm_{dev_sm})", flush=True)
        return "cuda"
    except Exception as e:
        print(f"[device] GPU init failed ({e}), using CPU", flush=True)
        return "cpu"

device = _pick_device()
print("device:", device, "| seed:", SEED, flush=True)

BLOCK_SIZE = 512   # units per sequence (fused kernel handles this fast on the rented GPU)
BATCH_SIZE = 32
MAX_STEPS = 3000   # full real run on the rented GPU (fast with the fused mamba-ssm kernel)
EVAL_EVERY = 200
LR = 3e-4
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
use_amp = (device == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def get_batch(train_arr, val_arr, split):
    data = train_arr if split == "train" else val_arr
    max_idx = len(data) - BLOCK_SIZE - 1
    if max_idx <= 0:
        raise ValueError(f"Dataset split has {len(data)} items, which is smaller than BLOCK_SIZE={BLOCK_SIZE}")
    ix = np.random.randint(0, max_idx, size=BATCH_SIZE)
    x = torch.stack([torch.from_numpy(data[i:i + BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + BLOCK_SIZE].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_val_loss(model, train_arr, val_arr, iters=20):
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(train_arr, val_arr, "val")
        with torch.cuda.amp.autocast(enabled=use_amp):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def bits_per_byte(nats_loss, bytes_per_unit=1.0):
    return (nats_loss / math.log(2)) / bytes_per_unit


def train_model(model, name, train_arr, val_arr, bytes_per_unit=1.0, max_steps=MAX_STEPS):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    history = {"step": [], "train_bpb": [], "val_bpb": [], "wall_clock_s": [], "peak_mem_mb": []}

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for step in range(1, max_steps + 1):
        x, y = get_batch(train_arr, val_arr, "train")
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            _, loss = model(x, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()

        if step % EVAL_EVERY == 0 or step == max_steps:
            val_loss = estimate_val_loss(model, train_arr, val_arr)
            elapsed = time.time() - t0
            peak_mem = torch.cuda.max_memory_allocated() / 1e6 if device == "cuda" else 0.0
            history["step"].append(step)
            history["train_bpb"].append(bits_per_byte(loss.item(), bytes_per_unit))
            history["val_bpb"].append(bits_per_byte(val_loss, bytes_per_unit))
            history["wall_clock_s"].append(elapsed)
            history["peak_mem_mb"].append(peak_mem)
            print(f"[{name}] step {step:5d} | train bpb {bits_per_byte(loss.item(), bytes_per_unit):.3f} "
                  f"| val bpb {bits_per_byte(val_loss, bytes_per_unit):.3f} | {elapsed:.1f}s | peak mem {peak_mem:.0f}MB", flush=True)

            torch.save({"model": model.state_dict(), "step": step, "history": history},
                       os.path.join(WORKING_DIR, f"{name}_ckpt.pt"))

    return history


In [ ]:
DATA_DIR = "/kaggle/working/data" if os.path.exists("/kaggle/working/data") else "./data"
byte_train = np.memmap(os.path.join(DATA_DIR, "train.bin"), dtype=np.uint8, mode="r")
byte_val = np.memmap(os.path.join(DATA_DIR, "val.bin"), dtype=np.uint8, mode="r")
tok_train_arr = np.memmap(os.path.join(DATA_DIR, "tok_train.bin"), dtype=np.int32, mode="r")
tok_val_arr = np.memmap(os.path.join(DATA_DIR, "tok_val.bin"), dtype=np.int32, mode="r")
print(f"byte train/val: {len(byte_train):,} / {len(byte_val):,} bytes")
print(f"token train/val: {len(tok_train_arr):,} / {len(tok_val_arr):,} tokens")


### 3b. Benchmark before committing to the full run

The Mamba scan is a sequential Python loop — its real speed depends on the
specific GPU/driver in use, not something to reason out from first
principles. Run this first (~20-40 steps per model, well under a minute
total) and read the extrapolated total against your actual available
compute budget before running Section 4 for real. Adjust `MAX_STEPS` above
and re-run this cell if the estimate doesn't fit.

In [ ]:
def benchmark(model_ctor, name, train_arr, val_arr, n_bench_steps=30):
    m = model_ctor().to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR)
    # warmup - first step(s) include CUDA context/kernel compilation, don't time those
    for _ in range(3):
        x, y = get_batch(train_arr, val_arr, "train")
        _, loss = m(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    for _ in range(n_bench_steps):
        x, y = get_batch(train_arr, val_arr, "train")
        _, loss = m(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    s_per_step = elapsed / n_bench_steps
    est_total_min = (s_per_step * MAX_STEPS) / 60
    print(f"[{name}] {s_per_step*1000:.1f} ms/step | "
          f"estimated total for {MAX_STEPS} steps: {est_total_min:.1f} min", flush=True)
    del m, opt
    if device == "cuda":
        torch.cuda.empty_cache()
    return s_per_step


print("Benchmarking all three models on real hardware (~1-2 min total)...\n", flush=True)
mamba_spb = benchmark(lambda: TinyMamba(d_model=256, n_layer=6, d_state=16, d_conv=4, expand=2),
                       "Mamba", byte_train, byte_val)
xf_byte_spb = benchmark(lambda: TinyTransformer(d_model=256, n_layer=6, n_head=8, max_len=BLOCK_SIZE, vocab_size=VOCAB_SIZE),
                         "Transformer-byte", byte_train, byte_val)
xf_tok_spb = benchmark(lambda: TinyTransformer(d_model=256, n_layer=6, n_head=8, max_len=BLOCK_SIZE, vocab_size=TOK_VOCAB_SIZE),
                        "Transformer-tokenized", tok_train_arr, tok_val_arr)

total_est_min = ((mamba_spb + xf_byte_spb + xf_tok_spb) * MAX_STEPS) / 60
print(f"\nEstimated total for ALL THREE models at MAX_STEPS={MAX_STEPS}: {total_est_min:.1f} min "
      f"({total_est_min/60:.1f} hours)", flush=True)
print("Compare this against your actual available compute budget. If it doesn't fit, "
      "lower MAX_STEPS in the cell above and re-run this benchmark to confirm.")

## 4. Run: train all three models

Sequential, not parallel, unless you confirm multiple GPUs are actually
allocated to this session.

In [ ]:
mamba_model = TinyMamba(d_model=256, n_layer=6, d_state=16, d_conv=4, expand=2)
mamba_history = train_model(mamba_model, "mamba", byte_train, byte_val, bytes_per_unit=1.0)

In [ ]:
xf_model = TinyTransformer(d_model=256, n_layer=6, n_head=8, max_len=BLOCK_SIZE, vocab_size=VOCAB_SIZE)
xf_history = train_model(xf_model, "transformer_byte", byte_train, byte_val, bytes_per_unit=1.0)

In [ ]:
actual_tok_vocab = sp.get_piece_size() if 'sp' in globals() else TOK_VOCAB_SIZE
xf_tok_model = TinyTransformer(d_model=256, n_layer=6, n_head=8, max_len=BLOCK_SIZE, vocab_size=actual_tok_vocab)
xf_tok_history = train_model(xf_tok_model, "transformer_tokenized", tok_train_arr, tok_val_arr,
                              bytes_per_unit=BYTES_PER_TOKEN)


## 5. Compare — this is the actual result

Validation bits-per-byte (all three models, same unit thanks to the
tokenized model's byte-conversion), wall-clock, and peak memory, at matched
parameters/steps. Small-scale, not converged — report it that way, not as a
competitive result.

How to read the three-way comparison for attribution:
- **Mamba (byte) vs. Transformer (byte)** isolates the architecture effect
- **Transformer (byte) vs. Transformer (tokenized)** isolates the tokenization effect
- **Mamba (byte) vs. Transformer (tokenized)** is the full comparison the original hypothesis is actually about

In [ ]:
import matplotlib.pyplot as plt

runs = [
    ("Mamba (byte)", mamba_history),
    ("Transformer (byte)", xf_history),
    ("Transformer (tokenized)", xf_tok_history),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for label, h in runs:
    axes[0].plot(h["step"], h["val_bpb"], label=label)
axes[0].set_xlabel("step"); axes[0].set_ylabel("val bits-per-byte"); axes[0].legend()
axes[0].set_title("Validation loss (lower = better)")

for label, h in runs:
    axes[1].plot(h["step"], h["wall_clock_s"], label=label)
axes[1].set_xlabel("step"); axes[1].set_ylabel("wall clock (s)"); axes[1].legend()
axes[1].set_title("Cumulative wall-clock time")

axes[2].bar([label for label, _ in runs], [max(h["peak_mem_mb"]) for _, h in runs])
axes[2].set_ylabel("peak GPU memory (MB)")
axes[2].set_title("Peak memory")
axes[2].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(WORKING_DIR, "comparison.png"), dpi=150)
plt.show()

print("Final val bits-per-byte, wall-clock, peak memory:")
for label, h in runs:
    print(f"  {label:26s} bpb={h['val_bpb'][-1]:.3f}  time={h['wall_clock_s'][-1]:.1f}s  "
          f"peak_mem={max(h['peak_mem_mb']):.0f}MB")

print("\nAttribution:")
print(f"  Architecture effect (byte Mamba vs byte Transformer):     "
      f"{xf_history['val_bpb'][-1] - mamba_history['val_bpb'][-1]:+.3f} bpb")
print(f"  Tokenization effect (byte Transformer vs tokenized Transformer): "
      f"{xf_tok_history['val_bpb'][-1] - xf_history['val_bpb'][-1]:+.3f} bpb")
print(f"  Full comparison (byte Mamba vs tokenized Transformer):    "
      f"{xf_tok_history['val_bpb'][-1] - mamba_history['val_bpb'][-1]:+.3f} bpb")

## 6. Entropy-vs-morphology comparison

Does the byte-level Mamba's own per-byte prediction entropy spike near real
Amharic morpheme boundaries, without ever being told what a morpheme is?
Uses HornMorpho (github.com/hltdi/HornMorpho) as ground truth. Install
verified this session: wheel-based pip install, then `hm.anal('a', word)`
returns segmentation/lemma/features per analysis.

This is a genuine qualitative pilot on a small hand-checked sample (not a
large-scale statistical claim) — label it that way in the write-up, that's
a description of what the method actually supports, not a hedge.

In [ ]:
# Install HornMorpho - wheel-based or source install
import os, glob, subprocess, sys
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
hm_dir = os.path.join(WORKING_DIR, "HornMorpho")
if not os.path.exists(hm_dir):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/hltdi/HornMorpho.git", hm_dir], check=True)

wheel_files = glob.glob(f"{hm_dir}/dist/*.whl")
if wheel_files:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", wheel_files[0]], check=True)
    print(f"Installed {wheel_files[0]}", flush=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", hm_dir], check=True)
    print(f"Installed HornMorpho from source at {hm_dir}", flush=True)


In [ ]:
import hm
hm.download('a')  # Amharic language data, first-use only

In [ ]:
@torch.no_grad()
def per_byte_entropy(model, byte_seq):
    model.eval()
    x = torch.tensor(byte_seq, dtype=torch.long, device=device).unsqueeze(0)
    logits, _ = model(x)
    probs = torch.softmax(logits, dim=-1)
    ent = -(probs * torch.log2(probs + 1e-9)).sum(-1)
    return ent.squeeze(0).cpu().numpy()


def get_hornmorpho_boundaries(word):
    """Returns the list of morpheme boundary strings HornMorpho finds for a word,
    or None if analysis failed (word not recognized, ambiguous parse, etc)."""
    try:
        analyses = hm.anal('a', word)
        if not analyses:
            return None
        return analyses[0].get('seg', None)
    except Exception as e:
        print(f"  [HornMorpho failed on '{word}']: {e}")
        return None


# Held-out sample from the Mamba model's validation data (byte_val, not the
# old val_data name - this was renamed when the data-loading cell was
# refactored to support both byte-level and tokenized data).
sample_bytes = byte_val[:400].tolist()
sample_text = bytes(sample_bytes).decode("utf-8", errors="replace")
entropies = per_byte_entropy(mamba_model, sample_bytes)

print("Sample text:")
print(sample_text)
print()

# Pull a handful of whitespace-separated words from the sample for the
# hand-checked comparison (Amharic uses space-separated words at the
# surface level, even though morphemes within a word are not space-marked).
candidate_words = [w.strip("፡።,.!?\"'()") for w in sample_text.split() if len(w) > 2][:20]

print(f"Checking {len(candidate_words)} words against HornMorpho ground truth:\n")
results = []
for word in candidate_words:
    boundaries = get_hornmorpho_boundaries(word)
    status = boundaries if boundaries else "[not recognized / no analysis]"
    print(f"  {word:20s} -> {status}")
    results.append((word, boundaries))

n_recognized = sum(1 for _, b in results if b)
print(f"\n{n_recognized}/{len(results)} words got a HornMorpho analysis.")
print("\nManual next step: for each recognized word, find its byte offset in "
      "sample_text, read off the entropy values at that span from `entropies` "
      "above, and check whether entropy peaks land near the '-' boundaries in "
      "the segmentation string. This is the qualitative pilot - report the "
      "count of words where they visibly align vs. don't, not a formal metric.")

## 7. Write-up checklist

- [ ] Hypothesis, stated precisely (byte-level Mamba vs. matched Transformer,
      with the tokenized-Transformer ablation isolating the two variables)
- [ ] Literature grounding — cite from `resources/` (papers already archived,
      see `resources/MANIFEST.md` for the organized index)
- [ ] Method: corpus sources + sizes, **cleaning/filtering stats** (Section 1's
      `clean_stats` printout - documents dropped for being too short,
      non-Amharic, or duplicates), all three model configs, param counts,
      step budget, seed
- [ ] Results: val bpb curves (all three, same unit), wall-clock, peak
      memory, and the architecture/tokenization attribution breakdown
      (Section 5's printed output)
- [ ] Qualitative result: entropy-vs-morphology pilot findings (Section 6),
      labeled as a small hand-checked sample, not a large-scale statistic
- [ ] Limitations, stated plainly: corpus size actually pulled (check the
      Section 1 printout - this varies run to run depending on which
      sources load), only exact-hash dedup done (no semantic-level
      deduplication or cluster-based quality filtering), short/non-converged
      training, no downstream task eval, single-seed runs (no variance
      estimate across seeds)
- [ ] Future work: further corpus scale-up, semantic/embedding-based data
      curation (MiniPile-style), the Hebbian episodic-memory continual-
      learning phase from earlier planning — labeled as *not yet executed*,
      not claimed as a result
- [ ] Reproducibility note: fixed seed (1337) across torch/numpy/CUDA, but
      cuDNN nondeterminism means this is practical, not bit-exact,
      reproducibility - state that precisely rather than overclaiming